# ✅ Soluzioni — Esercizi Guidati sugli Agenti AI
## Riferimento per: `04_esercizi_agenti_ai.ipynb`

---

> ⚠️ **Prima di aprire questo file, prova a risolvere gli esercizi in autonomia!**

Questo notebook contiene le soluzioni di riferimento per tutti e tre gli esercizi.
Ogni soluzione è annotata con commenti che spiegano le scelte implementative.

| Esercizio | Titolo | Stato |
|---|---|---|
| **1** | Il tuo primo Tool personalizzato | ✅ Soluzione fornita |
| **2** | Agente con tool di ricerca e calcolo | ✅ Soluzione fornita |
| **3** | Mini-agente conversazionale multi-turno | ✅ Soluzione fornita |

---
## ⚙️ Setup

In [ ]:
%pip install -q langchain langchain-ollama langchain-core

In [1]:
import os
import json
from typing import List, Optional
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool

MODEL_NAME = "llama3.2"  # oppure 'qwen2.5'
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL,
    temperature=0,
)

try:
    test_response = llm.invoke("Rispondi solo con: 'Pronto!'")
    print(f"✅ Connessione riuscita: {test_response.content.strip()}")
except Exception as e:
    print(f"❌ Errore di connessione: {e}")

✅ Connessione riuscita: 'Pronto!'


In [2]:
def mostra_conversazione(messaggi: List[BaseMessage]):
    print("=" * 70)
    print("📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE")
    print("=" * 70)
    for i, msg in enumerate(messaggi, 1):
        if isinstance(msg, SystemMessage):
            print(f"\n[{i}] ⚙️ SYSTEM:\n    {msg.content}")
        elif isinstance(msg, HumanMessage):
            print(f"\n[{i}] 👤 UTENTE:\n    {msg.content}")
        elif isinstance(msg, AIMessage):
            print(f"\n[{i}] 🤖 ASSISTENTE (AI):")
            if msg.content:
                print(f"    {msg.content}")
            if getattr(msg, 'tool_calls', None):
                for tc in msg.tool_calls:
                    print(f"    🛠️ [Tool Call] Funzione: '{tc['name']}' | Parametri: {json.dumps(tc['args'], ensure_ascii=False)}")
        elif isinstance(msg, ToolMessage):
            print(f"\n[{i}] 📥 RISULTATO TOOL (ID: {msg.tool_call_id}):\n    {msg.content}")
        else:
            print(f"\n[{i}] ❓ ALTRO ({type(msg).__name__}):\n    {msg.content}")
    print("\n" + "=" * 70)

---
## ✅ Soluzione Esercizio 1 — Il tuo primo Tool personalizzato

### Punti chiave della soluzione
- Il decoratore `@tool` trasforma una normale funzione in uno strumento utilizzabile dall'LLM.
- La **docstring** deve essere descrittiva: è quella che il modello legge per capire *quando* e *come* usare il tool.
- Dopo l'invocazione, il programma Python **esegue manualmente** il tool e restituisce il risultato come `ToolMessage`.

In [3]:
# ✅ SOLUZIONE 1 — Parte A: Definizione del tool conta_parole

@tool
def conta_parole(testo: str) -> str:
    """
    Conta il numero di parole in un testo fornito.

    Usa questo tool ogni volta che l'utente chiede di:
        - contare le parole in una frase o in un testo
        - sapere quante parole contiene un messaggio
        - verificare la lunghezza in parole di uno scritto

    Args:
        testo: La stringa di testo di cui contare le parole.

    Returns:
        Una stringa con il numero totale di parole trovate.
    """
    # split() divide per spazi e rimuove automaticamente gli spazi multipli
    num_parole = len(testo.split())
    return f"Il testo contiene {num_parole} parole."

print(f"✅ Tool '{conta_parole.name}' definito.")
print(f"Schema: {conta_parole.args}")

✅ Tool 'conta_parole' definito.
Schema: {'testo': {'title': 'Testo', 'type': 'string'}}


In [4]:
# ✅ SOLUZIONE 1 — Parte B: Collega il tool e ottieni la prima risposta

# Colleghiamo il tool al modello tramite bind_tools
llm_con_conta = llm.bind_tools([conta_parole])

# Messaggio utente che richiede il conteggio
frase_test = "L'intelligenza artificiale sta trasformando il mondo del lavoro"
prompt = f'Conta le parole nella seguente frase: "{frase_test}"'

messaggi_es1: List[BaseMessage] = [HumanMessage(content=prompt)]

# Prima chiamata: il modello decide di usare il tool
risposta = llm_con_conta.invoke(messaggi_es1)

print("Tipo risposta:", type(risposta).__name__)
print("Contenuto testuale:", risposta.content or "(nessun testo, ha generato una Tool Call)")
print("Tool calls:", risposta.tool_calls)

Tipo risposta: AIMessage
Contenuto testuale: (nessun testo, ha generato una Tool Call)
Tool calls: [{'name': 'conta_parole', 'args': {'testo': "L'intelligenza artificiale sta trasformando il mondo del lavoro"}, 'id': 'be3ba30a-327c-42fd-9305-e97bdded87fc', 'type': 'tool_call'}]


In [6]:
# ✅ SOLUZIONE 1 — Parte C: Esecuzione manuale del tool e risposta finale

# 1. Aggiungiamo la risposta AI (con la tool call) alla cronologia
messaggi_es1.append(risposta)

if risposta.tool_calls:
    for tc in risposta.tool_calls:
        nome_tool = tc["name"]
        argomenti = tc["args"]
        call_id = tc.get("id", "call_1")

        print(f"⚡ Esecuzione tool '{nome_tool}' con argomenti: {argomenti}")

        # 2. Eseguiamo la funzione Python corrispondente
        risultato = conta_parole.invoke(argomenti)
        print(f"📥 Risultato: {risultato}")

        # 3. Aggiungiamo il risultato come ToolMessage
        messaggi_es1.append(ToolMessage(content=str(risultato), tool_call_id=call_id))

    # 4. Richiesta finale al modello per formulare la risposta all'utente
    risposta_finale = llm_con_conta.invoke(messaggi_es1)
    messaggi_es1.append(risposta_finale)
    print(f"\n💬 Risposta finale dell'AI:\n{risposta_finale.content}")
else:
    print("Il modello ha risposto direttamente:", risposta.content)

print()
mostra_conversazione(messaggi_es1)

⚡ Esecuzione tool 'conta_parole' con argomenti: {'testo': "L'intelligenza artificiale sta trasformando il mondo del lavoro"}
📥 Risultato: Il testo contiene 8 parole.

💬 Risposta finale dell'AI:
La risposta è: La risposta è 8

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] 👤 UTENTE:
    Conta le parole nella seguente frase: "L'intelligenza artificiale sta trasformando il mondo del lavoro"

[2] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'conta_parole' | Parametri: {"testo": "L'intelligenza artificiale sta trasformando il mondo del lavoro"}

[3] 📥 RISULTATO TOOL (ID: be3ba30a-327c-42fd-9305-e97bdded87fc):
    Il testo contiene 8 parole.

[4] 🤖 ASSISTENTE (AI):
    La risposta è: La risposta è 8



---
## ✅ Soluzione Esercizio 2 — Agente con Tool di Ricerca e Calcolo

### Punti chiave della soluzione
- Il **catalogo** è un dizionario privato all'interno del tool: l'LLM non lo vede ma può interrogarlo tramite la funzione.
- La funzione `esegui_agente` implementa il **Reasoning Loop**: si ferma solo quando l'AI non genera più `tool_calls`.
- Il `map_tool` permette di trovare la funzione Python giusta dato il nome stringa restituito dall'LLM.

In [13]:
# ✅ SOLUZIONE 2 — Parte A: Definizione dei due tool

@tool
def cerca_prodotto(nome: str) -> str:
    """
    Cerca un prodotto nel catalogo e restituisce il suo prezzo base.

    Usa questo tool quando l'utente chiede informazioni sul prezzo di un prodotto o quando hai bisogno di conoscere il prezzo del prodotto per poi calcolare uno sconto.

    Args:
        nome: Il nome del prodotto da cercare (es. 'laptop', 'smartphone', 'tablet').

    Returns:
        Una stringa con il prezzo del prodotto oppure un messaggio di errore
        se il prodotto non è presente nel catalogo.
    """
    # Il catalogo è definito localmente nel tool (non accessibile dall'esterno)
    CATALOGO = {
        "laptop": 1200.0,
        "smartphone": 699.0,
        "tablet": 450.0,
        "cuffie": 89.0,
        "monitor": 320.0,
    }
    # Normalizziamo il nome per rendere la ricerca case-insensitive
    chiave = nome.lower().strip()
    if chiave in CATALOGO:
        return f"Prodotto trovato: '{chiave}' — Prezzo base: {CATALOGO[chiave]:.2f}€"
    else:
        prodotti_disponibili = ", ".join(CATALOGO.keys())
        return f"Prodotto '{nome}' non trovato nel catalogo. Prodotti disponibili: {prodotti_disponibili}"


@tool
def calcola_sconto(prezzo: float, percentuale_sconto: float) -> str:
    """
    Calcola il prezzo finale di un prodotto dopo l'applicazione di uno sconto percentuale.

    Usa questo tool per calcolare sconti su prezzi.

    Args:
        prezzo: Il prezzo originale del prodotto (in euro).
        percentuale_sconto: La percentuale di sconto da applicare (es. 15 per il 15%).

    Returns:
        Una stringa con il prezzo scontato e il risparmio in euro.
    """
    if not (0 <= percentuale_sconto <= 100):
        return f"Errore: la percentuale di sconto deve essere tra 0 e 100 (ricevuto: {percentuale_sconto})."

    risparmio = prezzo * (percentuale_sconto / 100)
    prezzo_finale = prezzo - risparmio
    return (
        f"Prezzo originale: {prezzo:.2f}€ | "
        f"Sconto {percentuale_sconto:.0f}%: -{risparmio:.2f}€ | "
        f"Prezzo finale: {prezzo_finale:.2f}€"
    )


print("✅ Tool 'cerca_prodotto' e 'calcola_sconto' definiti.")

✅ Tool 'cerca_prodotto' e 'calcola_sconto' definiti.


In [ ]:
# ✅ SOLUZIONE 2 — Parte B: Funzione esegui_agente (Reasoning Loop)

def esegui_agente(
    richiesta_utente: str,
    lista_tool: list,
    cronologia: Optional[List[BaseMessage]] = None,
    max_iterazioni: int = 6,
    verbose: bool = True,
) -> tuple[str, List[BaseMessage]]:
    """
    Esegue un loop agentico completo (Reasoning Loop / ReAct pattern).

    Il loop si chiude quando l'AI fornisce una risposta senza tool_calls,
    oppure al raggiungimento del numero massimo di iterazioni.

    Returns:
        Una tupla (risposta_finale_testo, cronologia_aggiornata)
    """
    # Creiamo il mappa nome_tool → funzione per l'esecuzione
    map_tool = {t.name: t for t in lista_tool}

    # Colleghiamo i tool al modello
    llm_con_tool = llm.bind_tools(lista_tool)

    # Inizializzazione cronologia: se non ne abbiamo una, partiamo da zero
    if cronologia is None:
        system_prompt = (
            "Usa un solo tool per volta. "
            "Dopo aver ricevuto l'output del tool, decidi il passo successivo. "
            "Non pianificare più tool contemporaneamente. Non chiedere spiegazioni ulteriore all'utente ma agisci."
        )
        messaggi: List[BaseMessage] = [SystemMessage(content=system_prompt)]
    else:
        # Copiamo la cronologia esistente per non mutare la lista originale
        messaggi = list(cronologia)

    # Aggiungiamo il nuovo messaggio dell'utente
    messaggi.append(HumanMessage(content=richiesta_utente))

    if verbose:
        print(f"🎯 NUOVA RICHIESTA: {richiesta_utente}\n" + "=" * 60)

    # ---- REASONING LOOP ----
    for step in range(1, max_iterazioni + 1):
        if verbose:
            print(f"\n📍 [Ciclo {step}] L'agente sta ragionando...")

        # Invocazione del modello con la cronologia corrente
        risposta = llm_con_tool.invoke(messaggi)
        messaggi.append(risposta)

        # Condizione di uscita: nessuna tool call → risposta finale
        if not risposta.tool_calls:
            if verbose:
                print("🏁 L'agente ha completato il task!")
            return risposta.content, messaggi

        # Esecuzione del primo tool richiesto (un tool per ciclo)
        tc = risposta.tool_calls[0]
        t_name = tc["name"]
        t_args = tc["args"]
        t_id = tc.get("id", f"call_{step}")

        if verbose:
            print(f"  ⚡ Tool '{t_name}' → argomenti: {t_args}")

        # Lookup nella mappa e invocazione
        if t_name in map_tool:
            risultato = map_tool[t_name].invoke(t_args)
        else:
            risultato = f"Errore: Tool '{t_name}' non trovato tra quelli disponibili."

        if verbose:
            print(f"  👁️ Osservazione: {risultato}")

        # Aggiungiamo il risultato come ToolMessage per il ciclo successivo
        messaggi.append(ToolMessage(content=str(risultato), tool_call_id=t_id))

    # Limite iterazioni raggiunto
    return "⚠️ Limite massimo di iterazioni raggiunto senza risposta finale.", messaggi


print("✅ Funzione 'esegui_agente' definita.")

✅ Funzione 'esegui_agente' definita.


In [18]:
# ✅ SOLUZIONE 2 — Parte C: Test dell'agente con i due tool


MODEL_NAME = "qwen2.5"  # Modello raccomandato per function calling locale (o 'qwen2.5')

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL,
    temperature=0,
)

lista_tool_es2 = [cerca_prodotto, calcola_sconto]

# Test 1: laptop con 15% di sconto
risposta_es2_1, storia_1 = esegui_agente(
    "Quanto costa un laptop dopo il 15% di sconto?",
    lista_tool_es2,
)
print("\n💬 Risposta finale:")
print(risposta_es2_1)

🎯 NUOVA RICHIESTA: Quanto costa un laptop dopo il 15% di sconto?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'cerca_prodotto' → argomenti: {'nome': 'laptop'}
  👁️ Osservazione: Prodotto trovato: 'laptop' — Prezzo base: 1200.00€

📍 [Ciclo 2] L'agente sta ragionando...
  ⚡ Tool 'calcola_sconto' → argomenti: {'prezzo': 1200, 'percentuale_sconto': 15}
  👁️ Osservazione: Prezzo originale: 1200.00€ | Sconto 15%: -180.00€ | Prezzo finale: 1020.00€

📍 [Ciclo 3] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta finale:
Il laptop, dopo il 15% di sconto, costa 1020.00€. Hai risparmiato 180.00€.


In [19]:
# Mostriamo la cronologia del primo test
mostra_conversazione(storia_1)

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] ⚙️ SYSTEM:
    Usa un solo tool per volta. Dopo aver ricevuto l'output del tool, decidi il passo successivo. Non pianificare più tool contemporaneamente. Non chiedere spiegazini ulteriore all'utente ma agisci.

[2] 👤 UTENTE:
    Quanto costa un laptop dopo il 15% di sconto?

[3] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'cerca_prodotto' | Parametri: {"nome": "laptop"}

[4] 📥 RISULTATO TOOL (ID: ac74ae44-3064-4b46-a57c-5c5f6c56d2af):
    Prodotto trovato: 'laptop' — Prezzo base: 1200.00€

[5] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'calcola_sconto' | Parametri: {"prezzo": 1200, "percentuale_sconto": 15}

[6] 📥 RISULTATO TOOL (ID: 34ae2f30-dce9-40f2-b464-6e4ce0a79ba6):
    Prezzo originale: 1200.00€ | Sconto 15%: -180.00€ | Prezzo finale: 1020.00€

[7] 🤖 ASSISTENTE (AI):
    Il laptop, dopo il 15% di sconto, costa 1020.00€. Hai risparmiato 180.00€.



In [ ]:
# Test 2: smartphone con 20% di sconto
risposta_es2_2, storia_2 = esegui_agente(
    "Qual è il prezzo di uno smartphone con il 20% di sconto?",
    lista_tool_es2,
)
print("\n💬 Risposta finale:")
print(risposta_es2_2)

---
## ✅ Soluzione Esercizio 3 — Mini-Agente Conversazionale Multi-Turno

### Punti chiave della soluzione
- Le note vengono salvate come file `.txt` nella cartella `note/`, rendendo l'agente **stateful** tra turni.
- Il tool `lista_note` non ha argomenti: usiamo uno schema con un parametro opzionale vuoto (o nessun parametro) per la compatibilità con il framework.
- La **memoria** si ottiene passando `cronologia=cronologia_completa` a ogni turno successivo — questo fa sì che l'agente ricordi tutto il contesto precedente.

In [20]:
# ✅ SOLUZIONE 3 — Parte A: Definizione dei tre tool per la gestione note

# Assicuriamoci che la cartella esista
os.makedirs("note", exist_ok=True)


@tool
def scrivi_nota(titolo: str, contenuto: str) -> str:
    """
    Salva una nota su disco con un titolo e un contenuto testuale.

    Usa questo tool ogni volta che l'utente chiede di:
        - salvare una nota
        - creare un appunto
        - memorizzare informazioni con un titolo

    Args:
        titolo: Il titolo della nota (diventa il nome del file, senza estensione).
        contenuto: Il testo da salvare nella nota.

    Returns:
        Messaggio di conferma del salvataggio oppure messaggio di errore.
    """
    try:
        percorso = f"note/{titolo}.txt"
        with open(percorso, "w", encoding="utf-8") as f:
            f.write(contenuto)
        return f"Nota '{titolo}' salvata con successo in '{percorso}' ({len(contenuto)} caratteri)."
    except Exception as e:
        return f"Errore durante il salvataggio della nota '{titolo}': {str(e)}"


@tool
def leggi_nota(titolo: str) -> str:
    """
    Legge il contenuto di una nota salvata sul disco.

    Usa questo tool quando l'utente vuole:
        - leggere una nota
        - recuperare il contenuto di un appunto
        - visualizzare le informazioni di una nota esistente

    Args:
        titolo: Il titolo della nota da leggere.

    Returns:
        Il contenuto della nota oppure un messaggio di errore se non trovata.
    """
    try:
        percorso = f"note/{titolo}.txt"
        if not os.path.exists(percorso):
            return f"Errore: La nota '{titolo}' non esiste. Controlla il titolo o usa 'lista_note' per vedere le note disponibili."
        with open(percorso, "r", encoding="utf-8") as f:
            contenuto = f.read()
        return f"Nota '{titolo}':\n{contenuto}"
    except Exception as e:
        return f"Errore durante la lettura della nota '{titolo}': {str(e)}"


@tool
def lista_note(filtro: str = "") -> str:
    """
    Elenca tutte le note salvate nella cartella 'note/'.

    Usa questo tool quando l'utente vuole:
        - vedere quante note esistono
        - ottenere l'elenco di tutti gli appunti salvati
        - sapere quali note sono disponibili

    Args:
        filtro: Stringa opzionale per filtrare le note per nome (lascia vuoto per tutte).

    Returns:
        L'elenco dei titoli delle note disponibili.
    """
    try:
        if not os.path.exists("note"):
            return "La cartella 'note' non esiste ancora. Nessuna nota salvata."

        tutti_i_file = os.listdir("note")
        # Filtriamo solo i file .txt e rimuoviamo l'estensione
        note = [f[:-4] for f in tutti_i_file if f.endswith(".txt")]

        # Applichiamo il filtro opzionale
        if filtro:
            note = [n for n in note if filtro.lower() in n.lower()]

        if not note:
            return "Nessuna nota trovata."

        elenco = "\n".join(f"  - {n}" for n in sorted(note))
        return f"Note disponibili ({len(note)} totali):\n{elenco}"
    except Exception as e:
        return f"Errore durante la lettura della cartella note: {str(e)}"


print("✅ Tool 'scrivi_nota', 'leggi_nota' e 'lista_note' definiti.")

✅ Tool 'scrivi_nota', 'leggi_nota' e 'lista_note' definiti.


In [21]:
# ✅ SOLUZIONE 3 — Parte B: Turno 1 — Scrivi una nota

lista_tool_es3 = [scrivi_nota, leggi_nota, lista_note]

richiesta_turno_1 = (
    "Scrivi una nota dal titolo 'Python' con questo contenuto: "
    "'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'"
)

risposta_t1, cronologia_completa = esegui_agente(
    richiesta_turno_1,
    lista_tool_es3,
)
print("\n💬 Risposta Turno 1:")
print(risposta_t1)

🎯 NUOVA RICHIESTA: Scrivi una nota dal titolo 'Python' con questo contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'scrivi_nota' → argomenti: {'titolo': 'Python', 'contenuto': 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'}
  👁️ Osservazione: Nota 'Python' salvata con successo in 'note/Python.txt' (85 caratteri).

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 1:
La tua nota dal titolo "Python" è stata salvata con successo. Contiene 85 caratteri di testo.


In [22]:
# ✅ SOLUZIONE 3 — Parte C: Turno 2 — Leggi la nota (con memoria del Turno 1)
# NOTA CHIAVE: passiamo cronologia_completa → l'agente ricorda il turno precedente

risposta_t2, cronologia_completa = esegui_agente(
    "Puoi leggere la nota che hai appena scritto su Python?",
    lista_tool_es3,
    cronologia=cronologia_completa,  # ← qui passiamo la memoria!
)
print("\n💬 Risposta Turno 2:")
print(risposta_t2)

🎯 NUOVA RICHIESTA: Puoi leggere la nota che hai appena scritto su Python?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'leggi_nota' → argomenti: {'titolo': 'Python'}
  👁️ Osservazione: Nota 'Python':
Python è un linguaggio di programmazione versatile, molto usato in AI e data science.

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 2:
Ecco il contenuto della tua nota "Python": "Python è un linguaggio di programmazione versatile, molto usato in AI e data science."


In [23]:
# ✅ SOLUZIONE 3 — Parte D: Turno 3 — Lista di tutte le note

risposta_t3, cronologia_completa = esegui_agente(
    "Quante note hai salvato finora? Elencale.",
    lista_tool_es3,
    cronologia=cronologia_completa,  # ← continuiamo ad accumulare la memoria
)
print("\n💬 Risposta Turno 3:")
print(risposta_t3)

🎯 NUOVA RICHIESTA: Quante note hai salvato finora? Elencale.

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'lista_note' → argomenti: {}
  👁️ Osservazione: Note disponibili (1 totali):
  - Python

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 3:
Hai salvato un totale di 1 nota. Ecco l'elenco delle note disponibili: Python.


In [ ]:
# Visualizzazione dell'intera cronologia dei 3 turni
print(f"Totale messaggi in cronologia: {len(cronologia_completa)}\n")
mostra_conversazione(messaggi=cronologia_completa)

Totale messaggi in cronologia: 13

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] ⚙️ SYSTEM:
    Usa un solo tool per volta. Dopo aver ricevuto l'output del tool, decidi il passo successivo. Non pianificare più tool contemporaneamente. Non chiedere spiegazini ulteriore all'utente ma agisci.

[2] 👤 UTENTE:
    Scrivi una nota dal titolo 'Python' con questo contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'

[3] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'scrivi_nota' | Parametri: {"titolo": "Python", "contenuto": "Python è un linguaggio di programmazione versatile, molto usato in AI e data science."}

[4] 📥 RISULTATO TOOL (ID: 63411c27-1b85-40af-a0da-3691e1e1bb4b):
    Nota 'Python' salvata con successo in 'note/Python.txt' (85 caratteri).

[5] 🤖 ASSISTENTE (AI):
    La tua nota dal titolo "Python" è stata salvata con successo. Contiene 85 caratteri di testo.

[6] 👤 UTENTE:
    Puoi leggere la nota che hai appena scritto su Python?

[

In [25]:
# Verifica finale: il file esiste davvero sul disco?
percorso_nota = "note/Python.txt"
if os.path.exists(percorso_nota):
    print(f"✅ Il file '{percorso_nota}' esiste!")
    with open(percorso_nota, "r", encoding="utf-8") as f:
        print("--- Contenuto ---")
        print(f.read())
        print("-----------------")
else:
    print(f"⚠️ Il file '{percorso_nota}' non è stato trovato.")

✅ Il file 'note/Python.txt' esiste!
--- Contenuto ---
Python è un linguaggio di programmazione versatile, molto usato in AI e data science.
-----------------


---
## 🏁 Riepilogo delle Soluzioni

| Esercizio | Concetti applicati | Pattern usato |
|---|---|---|
| **1 — Tool base** | `@tool`, docstring, type annotations, `ToolMessage` manuale | Tool Calling singolo |
| **2 — Multi-tool sequenziale** | Reasoning Loop, `map_tool`, tool chain (cerca → calcola) | ReAct multi-step |
| **3 — Multi-turno con memoria** | `cronologia` persistente tra turni, tool senza argomenti richiesti | Agente stateful conversazionale |

### 💡 Osservazioni finali

1. **La docstring è l'interfaccia tra te e l'LLM**: più è precisa e descrittiva, più il modello usa il tool correttamente.
2. **Un tool per ciclo**: il sistema prompt che limita a un tool per volta riduce le allucinazioni su modelli più piccoli.
3. **La memoria è nella lista messaggi**: passare la cronologia tra i turni è il meccanismo fondamentale per mantenere il contesto in un agente conversazionale.
4. **L'LLM non esegue codice**: il modello genera solo la *richiesta* di chiamata; è il tuo codice Python a invocare la funzione reale e a restituire l'osservazione tramite `ToolMessage`.